# Bilateral SDA campaign overview

Set `SDA_RUN_ROOT` before starting Jupyter, or edit `RUN_ROOT` below. The notebook discovers completed matrix builds, canonical SDA publications, effect sizes, validation results, and sourcing products without loading full effect arrays into memory. Results are accounting attributions, not causal responsibility.

In [ ]:
from pathlib import Path
from collections import defaultdict
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

RUN_ROOT = Path(os.environ.get('SDA_RUN_ROOT', '/local/s2053268/sda-gras-validation-2000-2001'))
SELECTED_KEY = os.environ.get('SDA_SELECTED_RUN', '2000_2001_2013')
assert RUN_ROOT.is_dir(), f'Run root not found: {RUN_ROOT}'
display(Markdown(f'**Run root:** `{RUN_ROOT}`  \n**Selected sourcing run:** `{SELECTED_KEY}`'))

In [ ]:
completion_path = RUN_ROOT / 'campaign_completion.json'
completion = json.loads(completion_path.read_text()) if completion_path.is_file() else {'status': 'campaign still running'}
display(pd.DataFrame([completion]).T.rename(columns={0: 'value'}))

build_rows = []
for path in sorted((RUN_ROOT / 'build').glob('*/pipeline_summary_*.json')):
    summary = json.loads(path.read_text())
    gras = summary.get('gras', {})
    build_rows.append({
        'year': path.parent.name,
        'accepted': gras.get('accepted'),
        'acceptance_status': gras.get('acceptance_status'),
        'iterations': gras.get('iterations'),
        'feasible_worst_ratio': gras.get('feasible_worst_threshold_ratio'),
        'infeasible_share': gras.get('infeasible_materiality_share'),
        'infeasible_rows': gras.get('n_sign_infeasible_rows'),
    })
builds = pd.DataFrame(build_rows)
display(Markdown('## Matrix-build status'))
display(builds if not builds.empty else Markdown('*No completed build summaries found.*'))

In [ ]:
TOP_LEVEL = ('D_v', 'D_C', 'D_A_x', 'D_YF', 'D_YB', 'D_y')
NESTED = ('D_T_coef', 'D_w_coef')

def mmap_metrics(path, chunk_size=1_000_000):
    values = np.load(path, mmap_mode='r').reshape(-1)
    net = positive = negative = 0.0
    for start in range(0, values.size, chunk_size):
        chunk = np.asarray(values[start:start + chunk_size], dtype=np.float64)
        net += float(chunk.sum(dtype=np.float64))
        positive += float(chunk[chunk > 0].sum(dtype=np.float64))
        negative += float(chunk[chunk < 0].sum(dtype=np.float64))
    return {'contribution': net, 'positive_contribution': positive,
            'negative_contribution': negative, 'gross_contribution': positive - negative}

def publication_summary(manifest_path):
    root = manifest_path.parent
    manifest = json.loads(manifest_path.read_text())
    metrics = {}
    for name in (*TOP_LEVEL, *NESTED):
        entry = manifest.get('effects', {}).get(name)
        if entry:
            metrics[name] = mmap_metrics(root / entry['baseline']['file'])
    totals = {name: values['contribution'] for name, values in metrics.items()}
    delta = mmap_metrics(root / manifest['total_change']['file'])['contribution']
    top_sum = sum(totals.get(name, 0.0) for name in TOP_LEVEL)
    nested_residual = totals.get('D_T_coef', np.nan) + totals.get('D_w_coef', np.nan) - totals.get('D_A_x', np.nan)
    parts = root.name.removeprefix('sda_').rsplit('_', 1)
    year_pair, resolution = parts if len(parts) == 2 else (root.name, '')
    row = {
        'run_id': root.name, 'year_pair': year_pair, 'resolution': resolution,
        'validation_passed': manifest.get('validation', {}).get('passed'),
        'delta_G': delta, 'top_level_residual': top_sum - delta,
        'Ax_nested_residual': nested_residual,
    }
    row.update(totals)
    detail = []
    start_year, end_year = (int(value) for value in year_pair.split('_'))
    for name, values in metrics.items():
        detail.append({'run_id': root.name, 'start_year': start_year, 'end_year': end_year,
                       'year_pair': year_pair, 'resolution': resolution, 'effect': name,
                       'validation_passed': row['validation_passed'], **values})
    return row, detail

manifest_paths = sorted((RUN_ROOT / 'sda').glob('*/manifest.json'))
summaries = [publication_summary(path) for path in manifest_paths]
effects = pd.DataFrame(item[0] for item in summaries)
annual = pd.DataFrame(row for item in summaries for row in item[1])
if not annual.empty:
    top_mask = annual['effect'].isin(TOP_LEVEL)
    gross_denominator = annual.loc[top_mask].groupby(['year_pair', 'resolution'])['gross_contribution'].transform('sum')
    delta_lookup = effects.set_index(['year_pair', 'resolution'])['delta_G']
    annual.loc[top_mask, 'gross_pct_of_all_effects'] = 100 * annual.loc[top_mask, 'gross_contribution'] / gross_denominator
    annual['net_pct_of_delta_G'] = [100 * row.contribution / delta_lookup.loc[(row.year_pair, row.resolution)]
                                    if delta_lookup.loc[(row.year_pair, row.resolution)] else np.nan
                                    for row in annual.itertuples()]
display(Markdown('## All completed annual contributions'))
display(annual.sort_values(['resolution', 'start_year', 'effect']) if not annual.empty
        else Markdown('*No canonical SDA manifests found yet.*'))
display(Markdown('### Publication closure overview'))
display(effects if not effects.empty else Markdown('*No canonical SDA manifests found yet.*'))

In [ ]:
if not annual.empty:
    completed = annual[annual['effect'].isin(TOP_LEVEL)].copy()
    resolutions = list(completed['resolution'].drop_duplicates())
    fig, axes = plt.subplots(len(resolutions), 1, figsize=(15, 4.5 * len(resolutions)), squeeze=False)
    for axis, resolution in zip(axes[:, 0], resolutions):
        panel = completed[completed['resolution'] == resolution].pivot(
            index='end_year', columns='effect', values='contribution'
        ).sort_index()
        panel.plot(ax=axis, marker='o')
        axis.axhline(0, color='black', linewidth=0.8)
        axis.set_title(f'Annual canonical effects — {resolution}')
        axis.set_xlabel('End year')
        axis.set_ylabel('Contribution (declared units)')
        axis.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    plt.show()

    display(Markdown('### Net contribution panel'))
    display(completed.pivot_table(index=['start_year', 'end_year'], columns=['resolution', 'effect'],
                                  values='contribution').sort_index())

In [ ]:
off = RUN_ROOT / 'offshoring' / SELECTED_KEY
report_paths = {
    'source_country': off / 'source_country' / 'source_country_cube_report.json',
    'source_group_sector': off / 'source_group_sector' / 'source_group_sector_cube_report.json',
}
report_rows = []
for product, path in report_paths.items():
    if path.is_file():
        report = json.loads(path.read_text())
        reconciliation = report.get('reconciliation', {})
        parent = reconciliation.get('parent', reconciliation)
        report_rows.append({
            'product': product, 'units': report.get('units'), 'shape': report.get('shape'),
            'total': report.get('total'), 'reconciled': parent.get('passed'),
            'max_abs_error': parent.get('max_abs_error'),
        })
display(Markdown(f'## Sourcing status: `{SELECTED_KEY}`'))
display(pd.DataFrame(report_rows) if report_rows else Markdown('*Sourcing products not complete yet.*'))

In [ ]:
def parquet_group_sum(path, keys, batch_size=100_000):
    import pyarrow.parquet as pq
    result = None
    parquet = pq.ParquetFile(path)
    for batch in parquet.iter_batches(batch_size=batch_size, columns=[*keys, 'Value']):
        grouped = batch.to_pandas().groupby(keys, dropna=False)['Value'].sum()
        result = grouped if result is None else result.add(grouped, fill_value=0.0)
    return result.sort_values(ascending=False) if result is not None else pd.Series(dtype=float)

country_path = off / 'source_country' / 'D_C.source_country_cube.parquet'
group_sector_path = off / 'source_group_sector' / 'D_C.source_group_sector_cube.parquet'
if country_path.is_file():
    source_country = parquet_group_sum(country_path, ['Source_Country'])
    display(Markdown('### Largest source-country contributions'))
    display(pd.concat({'largest_positive': source_country.head(15), 'largest_negative': source_country.tail(15).sort_values()} , axis=1))
if group_sector_path.is_file():
    source_group_sector = parquet_group_sum(group_sector_path, ['Source_Group', 'Source_Sector'])
    display(Markdown('### Largest source-group × sector contributions'))
    display(pd.concat({'largest_positive': source_group_sector.head(20), 'largest_negative': source_group_sector.tail(20).sort_values()}, axis=1))